In [1]:
import numpy as np
import pandas as pd
from datetime import timedelta

def gerar_dataset_sintetico_pd_behavior(
    *,
    n_contratos: int = 500,
    max_meses: int = 24,
    p_transicao_agreement: float = 0.20,
    p_transicao_billing: float = 0.08,
    p_transicao_reneg: float = 0.05,
    seed: int | None = 42,
) -> pd.DataFrame:
    """
    Gera um dataset de histórico de contratos contendo:
    --------------------------------------------------------------------------
    - ccb_paradigma : ID do contrato original
    - ccb           : ID do contrato "corrente" (muda após renegociação)
    - data_movimento: data da observação (1 ponto por mês)
    - produto       : status (LOAN, AGREEMENT_LOAN, LOAN_BILLING, RENEGOCIATION_LOAN)
    - atraso        : dias de atraso naquela data
    --------------------------------------------------------------------------

    Parâmetros
    ----------
    n_contratos : int
        Quantidade de contratos simulados.
    max_meses : int
        Duração máxima de acompanhamento em meses.
    p_transicao_agreement, p_transicao_billing, p_transicao_reneg : float
        Probabilidades mensais de mudar de LOAN para cada status respectivo.
        (Somente uma transição por contrato; valores devem fazer sentido juntos.)
    seed : int | None
        Semente para reprodutibilidade.

    Retorna
    -------
    DataFrame
        Histórico completo com ~ (n_contratos × max_meses) linhas, ordenado por
        ccb_paradigma e data_movimento.
    """
    rng = np.random.default_rng(seed)

    # ----------------------------------------------------------------------
    # 1 ▸ grade de datas mensais
    # ----------------------------------------------------------------------
    datas = pd.date_range("2023-01-01", periods=max_meses, freq="MS")

    linhas = []

    for cid in range(1, n_contratos + 1):
        ccb_paradigma = f"C{cid:06d}"
        ccb_corrente  = ccb_paradigma      # muda somente se RENEGOCIATION
        status_atual  = "LOAN"

        # flags para controlar se já houve transição
        transicionou   = False
        renegociado    = False

        for t, data in enumerate(datas):
            # Se já renegociou, nada mais muda
            if renegociado:
                # gera atraso menor (supõe contrato refinanciado)
                atraso = rng.integers(0, 15)
                linhas.append((ccb_paradigma, ccb_corrente, data, status_atual, atraso))
                continue

            # lógica de transição apenas enquanto estiver em LOAN
            if status_atual == "LOAN" and not transicionou:
                r = rng.random()
                if r < p_transicao_agreement:        # LOAN → AGREEMENT_LOAN
                    status_atual = "AGREEMENT_LOAN"
                    transicionou = True
                elif r < p_transicao_agreement + p_transicao_billing:   # LOAN → LOAN_BILLING
                    status_atual = "LOAN_BILLING"
                    transicionou = True
                elif r < (p_transicao_agreement + p_transicao_billing + p_transicao_reneg): # LOAN → RENEGOCIATION_LOAN
                    status_atual = "RENEGOCIATION_LOAN"
                    # muda ccb (novo contrato)
                    ccb_corrente = f"{ccb_paradigma}_R1"
                    renegociado = True
                    transicionou = True

            # Geração de atraso condicional ao status
            if status_atual == "LOAN":
                atraso = rng.integers(0, 30)                        # contrato regular
            elif status_atual == "AGREEMENT_LOAN":
                atraso = rng.integers(30, 120)                      # atraso alto à época do acordo
            elif status_atual == "LOAN_BILLING":
                atraso = rng.integers(60, 180)                      # cobrança
            else:  # RENEGOCIATION_LOAN
                atraso = rng.integers(0, 10)                        # novo contrato recém‐emitido

            linhas.append((ccb_paradigma, ccb_corrente, data, status_atual, atraso))

            # AGREEMENT_LOAN pode “falhar” depois de 3 meses e voltar a LOAN
            if status_atual == "AGREEMENT_LOAN" and t >= 2:
                if rng.random() < 0.35:          # 35 % voltam ao status LOAN
                    status_atual = "LOAN"
                    transicionou = False         # permite nova transição (simplificação)

    df = pd.DataFrame(
        linhas,
        columns=["ccb_paradigma", "ccb", "data_movimento", "produto", "atraso"]
    )
    return df.sort_values(["ccb_paradigma", "data_movimento"]).reset_index(drop=True)


In [2]:
import pandas as pd
from typing import Literal, Sequence

def atraso_antes_de_acordo(
    df: pd.DataFrame,
    *,
    id_col: str,                # coluna‑chave do contrato (ex.: "ccb_paradigma")
    status_col: str,            # coluna com o status (ex.: "produto")
    delay_col: str,             # coluna de dias em atraso (ex.: "atraso")
    date_col: str | None = None,# se precisar ordenar explicitamente
    status_loan: str = "LOAN",
    status_agreement: str = "AGREEMENT_LOAN",
    return_: Literal["series", "stats", "df"] = "series",
) -> pd.Series | pd.DataFrame:
    """
    Extrai, para cada contrato que passou de LOAN para AGREEMENT_LOAN,
    o valor de `delay_col` (dias de atraso) **na linha imediatamente
    anterior** ao primeiro status AGREEMENT_LOAN.

    Parameters
    ----------
    df : DataFrame
        Dados históricos dos contratos.
    id_col : str
        Nome da coluna que identifica o contrato *original* (ex.: "ccb_paradigma").
    status_col : str
        Nome da coluna que contém o status do contrato.
    delay_col : str
        Coluna numérica com dias de atraso (DPD).
    date_col : str or None, default=None
        Coluna temporal para ordenar os registros.  
        Se None, assume que o índice já está em ordem cronológica.
    status_loan, status_agreement : str
        Rótulos que identificam LOAN e AGREEMENT_LOAN, caso sejam diferentes.
    return_ : {"series", "stats", "df"}, default="series"
        - `"series"` → devolve `pd.Series` com um atraso por contrato.  
        - `"stats"`  → devolve `df.delay.describe()` (estatísticas).  
        - `"df"`     → devolve `DataFrame` com todas as colunas das linhas elegíveis.

    Returns
    -------
    Series | DataFrame
        Distribuição dos atrasos antes da 1ª transição LOAN→AGREEMENT para cada contrato.
    """

    # ------------------------------------------------------------------
    # 1. Ordenação cronológica dentro de cada contrato
    # ------------------------------------------------------------------
    if date_col:
        df = df.sort_values([id_col, date_col])
    else:
        df = df.sort_index()

    # ------------------------------------------------------------------
    # 2. Identificar linhas LOAN cujo próximo status = AGREEMENT_LOAN
    # ------------------------------------------------------------------
    df = df.copy()
    df["next_status"] = df.groupby(id_col)[status_col].shift(-1)

    mask_transition_loan_row = (
        (df[status_col]  == status_loan) &
        (df["next_status"] == status_agreement)
    )

    # ------------------------------------------------------------------
    # 3. Selecionar essas linhas (elas já são o ponto "imediatamente antes")
    # ------------------------------------------------------------------
    df_transitions = df.loc[mask_transition_loan_row]

    if return_ == "df":
        return df_transitions.drop(columns="next_status")

    delays = df_transitions[delay_col]

    if return_ == "stats":
        return delays.describe()

    return delays


In [4]:
df_mock.head(20)

,ccb_paradigma,ccb,data_movimento,produto,atraso
0,C000001,C000001,2023-01-01,LOAN,15
1,C000001,C000001,2023-02-01,AGREEMENT_LOAN,54
2,C000001,C000001,2023-03-01,AGREEMENT_LOAN,36
3,C000001,C000001,2023-04-01,AGREEMENT_LOAN,31
4,C000001,C000001,2023-05-01,AGREEMENT_LOAN,75
5,C000001,C000001,2023-06-01,AGREEMENT_LOAN,84
6,C000001,C000001,2023-07-01,AGREEMENT_LOAN,80
7,C000001,C000001,2023-08-01,AGREEMENT_LOAN,114
8,C000001,C000001,2023-09-01,LOAN,16
9,C000001,C000001,2023-10-01,LOAN,1


In [3]:
# 1) Gera o mock
df_mock = gerar_dataset_sintetico_pd_behavior(
    n_contratos=1000,          # pequeno para inspeção
    max_meses=24,
    seed=0
)

# 2) Usa sua função atraso_antes_de_acordo
delays = atraso_antes_de_acordo(
    df_mock,
    id_col     = "ccb_paradigma",
    status_col = "produto",
    delay_col  = "atraso",
    date_col   = "data_movimento"      # importante ordenar
)

print("Distribuição (dias de atraso) antes do 1º acordo:")
print(delays.describe())

Distribuição (dias de atraso) antes do 1º acordo:
count    978.000000
mean      14.760736
std        8.698522
min        0.000000
25%        7.000000
50%       15.000000
75%       22.000000
max       29.000000
Name: atraso, dtype: float64
